# Bonus: RAG (retrieval-augmented generation)

RAG: **retrieve** relevant chunks from your documents, then let the model **generate** an answer using them.

Pipeline here:

1. Load a PDF handbook.
2. Split into overlapping chunks.
3. Embed and store in an in-memory vector store.
4. Similarity-search, then wrap search as an agent tool.

Needs `OPENAI_API_KEY` (embeddings + chat).

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Semantic search

`PyPDFLoader` produces one `Document` per page. Printing `data` is noisy; it confirms load worked.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

`RecursiveCharacterTextSplitter` cuts text into ~1000-character chunks with 200-character overlap so sentences at boundaries are not lost. `add_start_index=True` stores the offset in metadata.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

An **embedding model** maps text to vectors so similar meaning is close in space.

Catalogue: [LangChain text embedding integrations](https://docs.langchain.com/oss/python/integrations/text_embedding)

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

# RAM-only index — fine for a demo PDF; use a persistent store in production
vector_store = InMemoryVectorStore(embeddings)

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

`similarity_search` embeds the query and returns nearest chunks. `results[0]` is the best match — not yet an LLM answer.

In [ ]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

## RAG agent

Expose search as a tool. The agent decides when to call it, then writes an answer from `page_content`.

In [ ]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [ ]:
from pprint import pprint

pprint(response)